In [1]:

!pip install basedosdados

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 16.0 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
import basedosdados as bd



In [4]:
PROJECT_ID = 'tcc-desastres-ambientais'

ANO_INICIO = 2015

ANO_FIM = 2025


query_inmet = f"""
SELECT
    ano,
    mes,
    data,
    id_estacao,
    SUM(precipitacao_total)   AS precipitacao_mm_dia,
    AVG(temperatura_max)      AS temperatura_max_media,
    AVG(umidade_rel_hora)     AS umidade_rel_media,
    AVG(vento_velocidade)     AS vento_velocidade_media,
    COUNT(*)                  AS leituras_validas
FROM `basedosdados.br_inmet_bdmep.microdados`
WHERE
    ano BETWEEN {ANO_INICIO} AND {ANO_FIM}
    AND precipitacao_total IS NOT NULL
    AND precipitacao_total >= 0
GROUP BY
    ano, mes, data, id_estacao
ORDER BY
    id_estacao, data
"""

print(' Extraindo dados do INMET...')
df_inmet = bd.read_sql(query_inmet, billing_project_id=PROJECT_ID)
print(f' Concluído! Shape: {df_inmet.shape}')
df_inmet.head()


 Extraindo dados do INMET...


BaseDosDadosAccessDeniedException: 
Are you sure you are using the right `billing_project_id`?
You must use the Project ID available in your Google Cloud console home page at https://console.cloud.google.com/home/dashboard
If you still don't have a Google Cloud Project, you have to create one.
You can set one up by following these steps: 
1. Go to this link https://console.cloud.google.com/projectselector2/home/dashboard
2. Agree with Terms of Service if asked
3. Click in Create Project
4. Put a cool name in your project
5. Hit create
6. Rerun this command with the flag `reauth=True`. 
   Like `read_table('br_ibge_pib', 'municipios', billing_project_id=<YOUR_PROJECT_ID>, reauth=True)`

In [ ]:
df_inmet.info()

In [ ]:
# Extrair tabela de estações completa
query_estacoes = """
SELECT
    id_municipio,
    id_estacao,
    estacao,
    data_fundacao,
    latitude,
    longitude,
    altitude
FROM `basedosdados.br_inmet_bdmep.estacao`
"""

df_estacoes = bd.read_sql(query_estacoes, billing_project_id=PROJECT_ID)
print(f'Estações: {df_estacoes.shape}')
df_estacoes.head()

In [ ]:
# JOIN — adiciona id_municipio, lat/lon no dataset principal
df_inmet = df_inmet.merge(
    df_estacoes[['id_estacao', 'id_municipio','estacao', 'latitude', 'longitude']],
    on='id_estacao',
    how='left'
)

print(f' Shape após JOIN: {df_inmet.shape}')
df_inmet.head()

In [ ]:
df_inmet.head()

In [ ]:


df_inmet.to_csv(
    '/content/drive/MyDrive/TCC/Data/Extract/20250101_inmet_pluviometria_brasil.csv',
    index=False,
    encoding='utf-8-sig'
)
print("Salvo!")